In [1]:
%cd ..

c:\Users\almei\Documents\GitHub\research_ner_leaderboard


In [2]:
from pathlib import Path
from transformers import AutoTokenizer
from transformers import AutoModelForTokenClassification

from spesia_ner.datasets import ClinicalRecordsDataset
from spesia_ner.autolabeling import AutoAnnotator


model_id = "best_model"
dataset_path = Path("data/Spesia/doccano/annotated_records")

tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=True)

ref_dataset = ClinicalRecordsDataset(dataset_path, tokenizer)
model = AutoModelForTokenClassification.from_pretrained(model_id, num_labels=ref_dataset.num_labels)
    
# Load dataset to be labeled
dataset_to_label_path = Path("data/Spesia/argilla/unique_unlabeled_records")

# Provide selected thresholds
best_thresholds = [0.79, 0.89, 0.07, 0.17, 0.07, 0.08, 0.9 , 0.14, 0.18, 0.19]

annotator = AutoAnnotator(
    tokenizer=tokenizer,
    model=model,
    best_thresholds=best_thresholds,
    idx_to_label=ref_dataset.idx_to_label,
    max_length=512,
    batch_size=25
)
annotated_dataset = annotator.annotate(dataset_to_label_path)

Loading all Records: 100%|██████████| 1/1 [00:00<00:00, 53.07it/s]
Annotating records: 100it [00:20,  4.82it/s]                       


In [3]:
ref_dataset.idx_to_label

{0: 'BRCA',
 1: 'DISTANT_METASTASIS',
 2: 'FISH_CISH_SISH',
 3: 'HER2',
 4: 'HISTOPATHOLOGICAL_TYPE',
 5: 'MENOPAUSE',
 6: 'METASTASIS_PLACE',
 7: 'RE',
 8: 'RP',
 9: 'SURGERY'}

In [4]:
annotated_dataset.export("20251104_argilla_data_auto_labeled.jsonl", format="jsonl", include_annotations=True)